# Medical RAG

This notebook is **self-contained**  It covers:

1. **Documents:** MedlinePlus + WHO + NHS + CDC pages.
2. **Evaluation:** 33 hand-written questions + auto-generated questions, retrieval metrics (Hit@3, MRR), refusal test, and **faithfulness** with an LLM judge plus a manual spot-check.
3. **Comparison:** `bge-small` vs `bge-base`, and Qwen 1.5B vs 3B vs 7B - all reported in tables.
*Educational project only, not medical advice.*

## Step 1: Install libraries

In [1]:
!pip -q install sentence-transformers rank_bm25 faiss-cpu bitsandbytes accelerate gradio beautifulsoup4 lxml requests pandas huggingface_hub

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.


## Step 2: Imports

In [2]:
import os, re, json, time, gc, random, requests
from types import SimpleNamespace
import numpy as np
import pandas as pd
import torch
import faiss
from bs4 import BeautifulSoup
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi

random.seed(42)
np.random.seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device, "| GPUs:", torch.cuda.device_count())

Device: cuda | GPUs: 2


## Step 3: Configuration

In [3]:
CONFIG = {
    # models to compare
    "EMBED_MODELS": ["BAAI/bge-small-en-v1.5", "BAAI/bge-base-en-v1.5"],
    "DEFAULT_EMBED": "BAAI/bge-small-en-v1.5",
    "RERANK_MODEL": "cross-encoder/ms-marco-MiniLM-L-6-v2",
    "LLM_MODELS": ["Qwen/Qwen2.5-1.5B-Instruct", "Qwen/Qwen2.5-3B-Instruct", "Qwen/Qwen2.5-7B-Instruct"],
    "JUDGE_MODEL": "Qwen/Qwen2.5-7B-Instruct",     # also writes the synthetic questions

    # chunking / retrieval
    "MAX_CHUNK_CHARS": 900,
    "OVERLAP_SENTENCES": 1,
    "TOP_K_CANDIDATES": 15,
    "TOP_K_FINAL": 3,
    "RERANK_THRESHOLD": -2.0,      # replaced automatically in Step 9

    # generation / evaluation
    "MAX_NEW_TOKENS": 300,
    "N_SYNTHETIC": 30,

    "WORK_DIR": "/kaggle/working",
}
os.makedirs(CONFIG["WORK_DIR"], exist_ok=True)

BGE_QUERY_PREFIX = "Represent this sentence for searching relevant passages: "
REFUSAL = "I don't know based on the provided documents."
DISCLAIMER = "General health information from public health websites, not a diagnosis. Please consult a doctor for medical advice."

## Step 4: Documents from MedlinePlus, WHO, NHS and CDC

Each entry is `(topic, organisation, url)`. The `topic` tag is used later to score retrieval across different websites.

- Some URLs may have moved or may block scripts (403). Failed pages are listed and skipped. For those, save the page as `.html`, upload the files as a Kaggle dataset and add them to `LOCAL_SOURCES`.
- Check each site's reuse terms (MedlinePlus and CDC: public health information, NHS: Open Government Licence, WHO: CC BY-NC-SA 3.0 IGO) and cite them in your thesis.

In [4]:
ML, WHO, NHS, CDC = "MedlinePlus", "WHO", "NHS", "CDC"

SOURCES = [
    # ---- MedlinePlus
    ("diabetes", ML, "https://medlineplus.gov/diabetes.html"),
    ("hypertension", ML, "https://medlineplus.gov/highbloodpressure.html"),
    ("asthma", ML, "https://medlineplus.gov/asthma.html"),
    ("heart disease", ML, "https://medlineplus.gov/heartdiseases.html"),
    ("obesity", ML, "https://medlineplus.gov/obesity.html"),
    ("cholesterol", ML, "https://medlineplus.gov/cholesterol.html"),
    ("stroke", ML, "https://medlineplus.gov/stroke.html"),
    ("copd", ML, "https://medlineplus.gov/copd.html"),
    ("kidney disease", ML, "https://medlineplus.gov/chronickidneydisease.html"),
    ("anemia", ML, "https://medlineplus.gov/anemia.html"),
    ("malaria", ML, "https://medlineplus.gov/malaria.html"),
    # ---- WHO fact sheets
    ("diabetes", WHO, "https://www.who.int/news-room/fact-sheets/detail/diabetes"),
    ("hypertension", WHO, "https://www.who.int/news-room/fact-sheets/detail/hypertension"),
    ("asthma", WHO, "https://www.who.int/news-room/fact-sheets/detail/asthma"),
    ("obesity", WHO, "https://www.who.int/news-room/fact-sheets/detail/obesity-and-overweight"),
    ("malaria", WHO, "https://www.who.int/news-room/fact-sheets/detail/malaria"),
    ("heart disease", WHO, "https://www.who.int/news-room/fact-sheets/detail/cardiovascular-diseases-(cvds)"),
    ("copd", WHO, "https://www.who.int/news-room/fact-sheets/detail/chronic-obstructive-pulmonary-disease-(copd)"),
    ("anemia", WHO, "https://www.who.int/news-room/fact-sheets/detail/anaemia"),
    # ---- NHS
    ("diabetes", NHS, "https://www.nhs.uk/conditions/type-2-diabetes/"),
    ("hypertension", NHS, "https://www.nhs.uk/conditions/high-blood-pressure-hypertension/"),
    ("asthma", NHS, "https://www.nhs.uk/conditions/asthma/"),
    ("heart disease", NHS, "https://www.nhs.uk/conditions/coronary-heart-disease/"),
    ("obesity", NHS, "https://www.nhs.uk/conditions/obesity/"),
    ("stroke", NHS, "https://www.nhs.uk/conditions/stroke/"),
    ("malaria", NHS, "https://www.nhs.uk/conditions/malaria/"),
    ("copd", NHS, "https://www.nhs.uk/conditions/chronic-obstructive-pulmonary-disease-copd/"),
    ("anemia", NHS, "https://www.nhs.uk/conditions/iron-deficiency-anaemia/"),
    # ---- CDC
    ("diabetes", CDC, "https://www.cdc.gov/diabetes/about/index.html"),
    ("hypertension", CDC, "https://www.cdc.gov/high-blood-pressure/about/index.html"),
    ("asthma", CDC, "https://www.cdc.gov/asthma/about/index.html"),
    ("heart disease", CDC, "https://www.cdc.gov/heart-disease/about/index.html"),
    ("stroke", CDC, "https://www.cdc.gov/stroke/about/index.html"),
    ("malaria", CDC, "https://www.cdc.gov/malaria/about/index.html"),
]

# Pages you saved yourself: (topic, organisation, path to .html file)
LOCAL_SOURCES = [
    # ("diabetes", "CDC", "/kaggle/input/my-medical-html/cdc_diabetes.html"),
]

HEADERS = {"User-Agent": "Mozilla/5.0 (academic medical RAG project)"}
PAGES, failed = [], []

for topic, org, url in SOURCES:
    try:
        r = requests.get(url, headers=HEADERS, timeout=30)
        r.raise_for_status()
        PAGES.append({"topic": topic, "org": org, "url": url, "html": r.text})
        print(f"OK     [{org:10s}] {url}")
    except Exception as e:
        failed.append((org, url, str(e)[:80]))
        print(f"FAILED [{org:10s}] {url} -> {str(e)[:80]}")
    time.sleep(0.5)

for topic, org, path in LOCAL_SOURCES:
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8", errors="ignore") as fh:
            PAGES.append({"topic": topic, "org": org, "url": os.path.basename(path), "html": fh.read()})
        print(f"LOCAL  [{org:10s}] {path}")
    else:
        print("Missing local file:", path)

print(f"\nLoaded {len(PAGES)} pages | failed {len(failed)}")
print(pd.Series([p["org"] for p in PAGES]).value_counts().to_string())

OK     [MedlinePlus] https://medlineplus.gov/diabetes.html
OK     [MedlinePlus] https://medlineplus.gov/highbloodpressure.html
OK     [MedlinePlus] https://medlineplus.gov/asthma.html
OK     [MedlinePlus] https://medlineplus.gov/heartdiseases.html
OK     [MedlinePlus] https://medlineplus.gov/obesity.html
OK     [MedlinePlus] https://medlineplus.gov/cholesterol.html
OK     [MedlinePlus] https://medlineplus.gov/stroke.html
OK     [MedlinePlus] https://medlineplus.gov/copd.html
OK     [MedlinePlus] https://medlineplus.gov/chronickidneydisease.html
OK     [MedlinePlus] https://medlineplus.gov/anemia.html
OK     [MedlinePlus] https://medlineplus.gov/malaria.html
OK     [WHO       ] https://www.who.int/news-room/fact-sheets/detail/diabetes
OK     [WHO       ] https://www.who.int/news-room/fact-sheets/detail/hypertension
OK     [WHO       ] https://www.who.int/news-room/fact-sheets/detail/asthma
OK     [WHO       ] https://www.who.int/news-room/fact-sheets/detail/obesity-and-overweight
OK    

## Step 5: Clean sections and build chunks

Same section-aware chunking as Part 1, with a generic container search that works across the four sites, plus a filter that drops boilerplate sections (references, share, feedback...). **Check the per-page chunk counts:** a page with fewer than 3 sections probably needs a custom selector.

In [5]:
SKIP_HEADINGS = re.compile(r"^(references?|related|share|feedback|cookies?|page last|sources?|more information|"
                           r"find a|contact|follow|subscribe|print|download)", re.I)

def extract_sections(html, fallback_title):
    soup = BeautifulSoup(html, "lxml")
    title = (soup.title.get_text(strip=True) if soup.title else fallback_title)
    title = re.split(r"\s[|\-–]\s", title)[0].strip()

    for tag in soup(["script", "style", "nav", "header", "footer", "aside", "form", "noscript", "svg", "button"]):
        tag.decompose()

    def collect(root):
        sections, heading, buf = [], "Overview", []

        def flush():
            text = re.sub(r"\s+", " ", " ".join(buf)).strip()
            if len(text) > 80 and not SKIP_HEADINGS.match(heading):
                sections.append((heading, text))

        for el in root.find_all(["h1", "h2", "h3", "h4", "p", "li"]):
            if el.name in ("h1", "h2", "h3", "h4"):
                flush()
                buf.clear()
                heading = el.get_text(" ", strip=True)
            elif el.find_parent("li") and el.name in ("p", "li"):
                continue
            else:
                buf.append(el.get_text(" ", strip=True))
        flush()
        return sections

    for selector in ["#topic-summary", "main article", "article", "main", "body"]:
        root = soup.select_one(selector)
        if root is None:
            continue
        sections = collect(root)
        if len(sections) >= 3 or selector == "body":
            return title, sections
    return title, []


def split_text(text, max_chars, overlap_sentences):
    sents = re.split(r"(?<=[.!?])\s+", text)
    out, cur = [], []
    for s in sents:
        if cur and len(" ".join(cur)) + len(s) > max_chars:
            out.append(" ".join(cur))
            cur = cur[-overlap_sentences:] if overlap_sentences else []
        cur.append(s)
    if cur:
        out.append(" ".join(cur))
    return out


CHUNKS, seen = [], set()
for p in PAGES:
    title, sections = extract_sections(p["html"], p["url"])
    n0 = len(CHUNKS)
    for heading, text in sections:
        for piece in split_text(text, CONFIG["MAX_CHUNK_CHARS"], CONFIG["OVERLAP_SENTENCES"]):
            if piece in seen:
                continue
            seen.add(piece)
            CHUNKS.append({
                "id": len(CHUNKS), "topic": p["topic"], "org": p["org"],
                "title": title, "heading": heading, "url": p["url"], "text": piece,
                "embed_text": f"{title} - {heading}\n{piece}",
            })
    n = len(CHUNKS) - n0
    flag = "  <-- CHECK (few chunks)" if n < 3 else ""
    print(f"[{p['org']:10s}] {p['topic']:15s} sections={len(sections):3d} chunks={n:3d}{flag}")

TOPICS_PRESENT = {c["topic"] for c in CHUNKS}
print("\nTotal chunks:", len(CHUNKS), "| topics:", sorted(TOPICS_PRESENT))
print(pd.Series([c["org"] for c in CHUNKS]).value_counts().to_string())

[MedlinePlus] diabetes        sections=  8 chunks= 10
[MedlinePlus] hypertension    sections=  5 chunks=  5
[MedlinePlus] asthma          sections=  6 chunks=  8
[MedlinePlus] heart disease   sections=  8 chunks=  9
[MedlinePlus] obesity         sections=  5 chunks=  7
[MedlinePlus] cholesterol     sections=  7 chunks= 10
[MedlinePlus] stroke          sections=  7 chunks=  9
[MedlinePlus] copd            sections=  8 chunks= 11
[MedlinePlus] kidney disease  sections= 12 chunks= 14
[MedlinePlus] anemia          sections= 14 chunks= 15
[MedlinePlus] malaria         sections=  7 chunks=  7
[WHO       ] diabetes        sections= 10 chunks= 13
[WHO       ] hypertension    sections=  9 chunks= 15
[WHO       ] asthma          sections=  8 chunks= 12
[WHO       ] obesity         sections= 11 chunks= 21
[WHO       ] malaria         sections= 15 chunks= 20
[WHO       ] heart disease   sections=  9 chunks= 18
[WHO       ] copd            sections=  7 chunks= 14
[WHO       ] anemia          sectio

## Step 6: Embedding models, FAISS indexes, BM25 and reranker

In [6]:
def tokenize(text):
    return re.findall(r"[a-z0-9]+", text.lower())

bm25 = BM25Okapi([tokenize(c["embed_text"]) for c in CHUNKS])
reranker = CrossEncoder(CONFIG["RERANK_MODEL"], device=device)

def build_kb(embed_name):
    kb = SimpleNamespace(name=embed_name)
    kb.embedder = SentenceTransformer(embed_name, device=device)
    emb = kb.embedder.encode(
        [c["embed_text"] for c in CHUNKS], batch_size=32, normalize_embeddings=True,
        convert_to_numpy=True, show_progress_bar=False,
    ).astype("float32")
    kb.index = faiss.IndexFlatIP(emb.shape[1])
    kb.index.add(emb)
    return kb

KBS = {}
for name in CONFIG["EMBED_MODELS"]:
    KBS[name] = build_kb(name)
    print("Built index for", name, "| vectors:", KBS[name].index.ntotal, "| dim:", KBS[name].index.d)

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

... (26 outputs left out)

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Built index for BAAI/bge-base-en-v1.5 | vectors: 339 | dim: 768


## Step 7: Retrieval function (dense / hybrid / hybrid + rerank)

In [7]:
def retrieve(query, kb, mode="hybrid+rerank", k_final=None, k_cand=None):
    k_final = k_final or CONFIG["TOP_K_FINAL"]
    k_cand = k_cand or CONFIG["TOP_K_CANDIDATES"]

    q = kb.embedder.encode([BGE_QUERY_PREFIX + query], normalize_embeddings=True).astype("float32")
    sims, ids = kb.index.search(q, k_cand)
    dense = [(int(i), float(s)) for i, s in zip(ids[0], sims[0]) if i >= 0]
    if mode == "dense":
        return [{"idx": i, "score": s} for i, s in dense[:k_final]]

    bm_scores = bm25.get_scores(tokenize(query))
    bm_ids = np.argsort(-bm_scores)[:k_cand]
    rrf = {}
    for rank, (i, _) in enumerate(dense):
        rrf[i] = rrf.get(i, 0.0) + 1.0 / (60 + rank + 1)
    for rank, i in enumerate(bm_ids):
        rrf[int(i)] = rrf.get(int(i), 0.0) + 1.0 / (60 + rank + 1)
    fused = sorted(rrf, key=rrf.get, reverse=True)[:k_cand]
    if mode == "hybrid":
        return [{"idx": i, "score": rrf[i]} for i in fused[:k_final]]

    scores = reranker.predict([(query, CHUNKS[i]["embed_text"]) for i in fused])
    order = np.argsort(-scores)[:k_final]
    return [{"idx": fused[o], "score": float(scores[o])} for o in order]

for r in retrieve("What are the symptoms of malaria?", KBS[CONFIG["DEFAULT_EMBED"]]):
    c = CHUNKS[r["idx"]]
    print(f"{r['score']:.2f} | {c['org']} | {c['title'][:30]} > {c['heading'][:40]}")

8.94 | NHS | Malaria > Symptoms of malaria
8.28 | WHO | Malaria > Symptoms
7.45 | WHO | Malaria > Overview


## Step 8: Evaluation questions

- **Hand-written set:** 3 questions per topic (33 total) with expected topic + keywords. A retrieval hit means a top-k chunk is from the right topic **and** contains one of the keywords. Adjust keywords if your pages use different wording.
- **Calibration questions** (used only to set the refusal threshold) are kept separate from the **test questions** (used only for reporting), so the threshold is not tuned on the test set.

In [8]:
HAND_EVAL = [
    {"q": "What are the symptoms of diabetes?", "topic": "diabetes", "kw": ["thirst", "urinat"]},
    {"q": "How is diabetes diagnosed?", "topic": "diabetes", "kw": ["a1c", "glucose", "blood sugar", "blood test"]},
    {"q": "What are the types of diabetes?", "topic": "diabetes", "kw": ["type 1", "type 2"]},
    {"q": "How is high blood pressure treated?", "topic": "hypertension", "kw": ["medicine", "medication", "lifestyle"]},
    {"q": "What are the complications of high blood pressure?", "topic": "hypertension", "kw": ["heart attack", "stroke", "kidney"]},
    {"q": "How can I lower my blood pressure?", "topic": "hypertension", "kw": ["salt", "exercise", "physical activity", "weight", "diet"]},
    {"q": "What triggers asthma attacks?", "topic": "asthma", "kw": ["trigger", "allergen", "dust", "pollen"]},
    {"q": "What are the symptoms of asthma?", "topic": "asthma", "kw": ["wheez", "cough", "breath"]},
    {"q": "How is asthma treated?", "topic": "asthma", "kw": ["inhaler", "medicine", "reliever", "controller"]},
    {"q": "What are the risk factors for heart disease?", "topic": "heart disease", "kw": ["risk"]},
    {"q": "What are the symptoms of heart disease?", "topic": "heart disease", "kw": ["chest", "symptom"]},
    {"q": "How can heart disease be prevented?", "topic": "heart disease", "kw": ["healthy", "prevent", "exercise"]},
    {"q": "How is obesity diagnosed?", "topic": "obesity", "kw": ["body mass index", "bmi"]},
    {"q": "What causes obesity?", "topic": "obesity", "kw": ["calorie", "eat", "cause"]},
    {"q": "How can obesity be prevented?", "topic": "obesity", "kw": ["healthy", "activity", "prevent", "diet"]},
    {"q": "What is high cholesterol?", "topic": "cholesterol", "kw": ["cholesterol"]},
    {"q": "How can I lower my cholesterol?", "topic": "cholesterol", "kw": ["diet", "statin", "exercise", "medicine"]},
    {"q": "What are the risks of high cholesterol?", "topic": "cholesterol", "kw": ["heart", "artery", "risk"]},
    {"q": "What are the warning signs of a stroke?", "topic": "stroke", "kw": ["sudden", "numbness", "face", "speech"]},
    {"q": "What are the types of stroke?", "topic": "stroke", "kw": ["ischemic", "hemorrhagic", "haemorrhagic", "clot"]},
    {"q": "How is a stroke treated?", "topic": "stroke", "kw": ["clot", "medicine", "surgery", "emergency", "rehabilitation"]},
    {"q": "What is COPD?", "topic": "copd", "kw": ["copd", "lung"]},
    {"q": "What are the symptoms of COPD?", "topic": "copd", "kw": ["cough", "breath"]},
    {"q": "What causes COPD?", "topic": "copd", "kw": ["smok", "cause"]},
    {"q": "How is chronic kidney disease diagnosed?", "topic": "kidney disease", "kw": ["blood test", "urine", "gfr"]},
    {"q": "What are the symptoms of kidney disease?", "topic": "kidney disease", "kw": ["symptom", "tired", "swelling", "urin"]},
    {"q": "How is chronic kidney disease treated?", "topic": "kidney disease", "kw": ["treat", "dialysis", "medicine", "blood pressure"]},
    {"q": "What are the symptoms of anemia?", "topic": "anemia", "kw": ["tired", "weak", "fatigue", "pale"]},
    {"q": "What causes anemia?", "topic": "anemia", "kw": ["iron", "cause", "blood loss", "vitamin"]},
    {"q": "How is anemia treated?", "topic": "anemia", "kw": ["iron", "treat", "supplement", "diet"]},
    {"q": "What are the symptoms of malaria?", "topic": "malaria", "kw": ["fever", "chill"]},
    {"q": "How is malaria transmitted?", "topic": "malaria", "kw": ["mosquito"]},
    {"q": "How can malaria be prevented?", "topic": "malaria", "kw": ["net", "prevent", "mosquito", "medicine"]},
]
dropped = [x["q"] for x in HAND_EVAL if x["topic"] not in TOPICS_PRESENT]
HAND_EVAL = [x for x in HAND_EVAL if x["topic"] in TOPICS_PRESENT]
print("Hand-written questions kept:", len(HAND_EVAL), "| dropped (topic not loaded):", len(dropped))

# calibration only
IN_CAL = ["What foods should I avoid with diabetes?", "Can asthma be cured?", "What is a heart attack?",
          "What is BMI?", "How does smoking affect the lungs?", "What is dialysis?", "What does LDL mean?",
          "What is hypertension?", "What medicines treat malaria?", "What is iron deficiency?"]
OUT_CAL = ["What is the weather like today?", "How do I fix a flat bike tire?", "Who is the president of France?",
           "How do I learn guitar?", "What is the price of gold?", "How do I bake bread?"]

# test only (never used for calibration)
OUT_EVAL = ["How do I treat a broken arm?", "Who won the football world cup?", "What is the capital of France?",
            "Write a python function to sort a list.", "What is the best smartphone to buy?",
            "How do I cook biryani?", "What are the symptoms of dengue fever?", "How is tuberculosis treated?"]

Hand-written questions kept: 33 | dropped (topic not loaded): 0


## Step 9: Calibrate the refusal threshold

Picks the rerank-score cut-off that best separates in-scope from out-of-scope **calibration** questions (balanced accuracy).

In [9]:
kb0 = KBS[CONFIG["DEFAULT_EMBED"]]
best_score = lambda q, kb=kb0: retrieve(q, kb, "hybrid+rerank")[0]["score"]

s_in = [best_score(q) for q in IN_CAL]
s_out = [best_score(q) for q in OUT_CAL]
print("In-scope  scores:", [round(s, 1) for s in s_in])
print("Out-scope scores:", [round(s, 1) for s in s_out])

def bal_acc(t):
    tpr = np.mean([s >= t for s in s_in])
    tnr = np.mean([s < t for s in s_out])
    return (tpr + tnr) / 2

cands = sorted(set(s_in + s_out))
thresholds = [(a + b) / 2 for a, b in zip(cands[:-1], cands[1:])]
top = max(bal_acc(t) for t in thresholds)
best_ts = [t for t in thresholds if bal_acc(t) == top]
CONFIG["RERANK_THRESHOLD"] = round(float(np.median(best_ts)), 2)
print(f"\nBalanced accuracy {top:.2f} -> RERANK_THRESHOLD set to {CONFIG['RERANK_THRESHOLD']}")
print("(override manually here if you prefer, e.g. CONFIG['RERANK_THRESHOLD'] = -3.0)")

In-scope  scores: [1.6, 9.0, 3.4, 2.6, 2.9, -4.3, 5.7, 4.7, 8.3, 3.2]
Out-scope scores: [-9.6, -11.1, -10.9, -10.9, -11.0, -10.6]

Balanced accuracy 1.00 -> RERANK_THRESHOLD set to -6.94
(override manually here if you prefer, e.g. CONFIG['RERANK_THRESHOLD'] = -3.0)


## Step 10: LLM wrapper, prompts and the full pipeline

All LLMs are loaded in 4-bit so that the comparison between sizes is fair and fits a T4. `LocalLLM.close()` frees GPU memory before the next model is loaded.

In [10]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

class LocalLLM:
    def __init__(self, name):
        self.name = name
        self.tok = AutoTokenizer.from_pretrained(name)
        kwargs = {"device_map": "auto"}
        if torch.cuda.is_available():
            kwargs["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
        try:
            self.model = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float16, **kwargs)
        except TypeError:
            self.model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.float16, **kwargs)
        self.model.eval()

    def chat(self, messages, max_new_tokens=None):
        max_new_tokens = max_new_tokens or CONFIG["MAX_NEW_TOKENS"]
        text = self.tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.tok(text, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            out = self.model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                                      temperature=None, top_p=None, top_k=None,
                                      pad_token_id=self.tok.eos_token_id)
        return self.tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

    def close(self):
        del self.model, self.tok
        gc.collect()
        torch.cuda.empty_cache()


EMERGENCY_PATTERN = re.compile(
    r"chest pain|can'?t breathe|cannot breathe|difficulty breathing|shortness of breath|"
    r"stroke|face droop|sudden weakness|unconscious|seizure|severe bleeding|suicid|overdose", re.I)
EMERGENCY_NOTE = ("**If this is happening to you or someone else right now, contact your local emergency "
                  "number or go to the nearest emergency department immediately.**")

SYSTEM_PROMPT = (
    "You are a careful medical information assistant.\n"
    "Answer the user's question using ONLY the numbered context passages provided.\n"
    "Rules:\n"
    "1. Do not use outside knowledge and do not add facts that are not in the context.\n"
    "2. Be concise: a short paragraph or a few bullet points.\n"
    "3. Mention passage numbers like [1] or [2] after the facts they support.\n"
    f"4. If the context does not contain the answer, reply exactly: {REFUSAL}\n"
    "5. Never diagnose the user or prescribe doses; suggest talking to a doctor when relevant."
)

def rewrite_question(llm, question, history):
    if not history:
        return question
    convo = "\n".join(f"User: {u}\nAssistant: {a[:300]}" for u, a in history[-3:])
    msgs = [
        {"role": "system", "content": "Rewrite the user's last question as a single standalone question that makes "
                                      "sense without the conversation. Replace pronouns like 'it' or 'its' with the "
                                      "topic. If it is already standalone, return it unchanged. Output only the question."},
        {"role": "user", "content": f"Conversation:\n{convo}\n\nLast question: {question}\n\nStandalone question:"},
    ]
    out = llm.chat(msgs, max_new_tokens=60).split("\n")[0].strip()
    return out or question


def ask(question, kb, llm, history=None):
    history = history or []
    standalone = rewrite_question(llm, question, history)
    results = retrieve(standalone, kb, "hybrid+rerank")
    best = results[0]["score"] if results else float("-inf")
    out = {"question": question, "standalone": standalone, "best_score": best, "answer": REFUSAL,
           "sources": [], "answered": False, "context": ""}

    if best < CONFIG["RERANK_THRESHOLD"]:
        if EMERGENCY_PATTERN.search(question):
            out["answer"] = REFUSAL + "\n\n" + EMERGENCY_NOTE
        return out

    context = "\n\n".join(
        f"[{n}] ({CHUNKS[r['idx']]['org']}: {CHUNKS[r['idx']]['title']} - {CHUNKS[r['idx']]['heading']})\n{CHUNKS[r['idx']]['text']}"
        for n, r in enumerate(results, start=1))
    out["context"] = context
    answer = llm.chat([{"role": "system", "content": SYSTEM_PROMPT},
                       {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {standalone}"}])

    if REFUSAL.lower() in answer.lower() and len(answer) < len(REFUSAL) + 20:
        out["answer"] = REFUSAL
    else:
        out["answer"], out["answered"] = answer, True
        seen_keys = set()
        for n, r in enumerate(results, start=1):
            c = CHUNKS[r["idx"]]
            key = (c["url"], c["heading"])
            if key in seen_keys:
                continue
            seen_keys.add(key)
            out["sources"].append({"n": n, "org": c["org"], "title": c["title"], "heading": c["heading"],
                                   "url": c["url"], "score": round(r["score"], 2)})
    if EMERGENCY_PATTERN.search(question):
        out["answer"] += "\n\n" + EMERGENCY_NOTE
    return out
print("Pipeline defined.")

Pipeline defined.


## Step 11: Synthetic questions (to grow the eval set past 50)

The judge LLM writes one patient-style question for each of `N_SYNTHETIC` random chunks. The **ground truth is that chunk id**, so no manual keyword labelling is needed. The set is cached to disk.

**Caveats for your thesis:** generated questions reuse words from their chunk, so they are easier than real questions, and another chunk (for example from a different website) may also answer them, so strict chunk-id hits can understate retrieval. Report the hand-written and synthetic results separately.

In [11]:
SYN_PATH = os.path.join(CONFIG["WORK_DIR"], "synthetic_eval.json")

if os.path.exists(SYN_PATH):
    SYNTH = json.load(open(SYN_PATH, encoding="utf-8"))
    print("Loaded cached synthetic questions:", len(SYNTH))
else:
    judge = LocalLLM(CONFIG["JUDGE_MODEL"])
    pool = [c["id"] for c in CHUNKS if len(c["text"]) >= 300]
    random.shuffle(pool)
    SYNTH = []
    for cid in tqdm(pool):
        if len(SYNTH) >= CONFIG["N_SYNTHETIC"]:
            break
        q = judge.chat([{"role": "user", "content":
            "Write ONE question that a patient might ask, which is answered by the text below. "
            "Do not mention 'the text' or 'the passage'. Output only the question.\n\n" + CHUNKS[cid]["text"]}],
            max_new_tokens=50).split("\n")[0].strip()
        if q.endswith("?") and 20 < len(q) < 200:
            SYNTH.append({"q": q, "chunk_id": cid})
    judge.close()
    json.dump(SYNTH, open(SYN_PATH, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    print("Generated and saved:", len(SYNTH))

for s in SYNTH[:5]:
    print("-", s["q"])
print("\nTotal eval questions (hand + synthetic):", len(HAND_EVAL) + len(SYNTH))

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

... (2 outputs left out)

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

  0%|          | 0/249 [00:00<?, ?it/s]

Generated and saved: 30
- What kind of tests might be done to diagnose a potential stroke?
- What conditions related to anemia are discussed on the Nemours Foundation website?
- When should I seek emergency medical help for my breathing issues?
- What are some of the health risks associated with overweight and obesity, especially in children and adolescents?
- What are some of the factors contributing to the global challenge of obesity?

Total eval questions (hand + synthetic): 63


## Step 12: Retrieval comparison - embedding model x retriever

Reports Hit@3 and MRR for `bge-small` vs `bge-base`, each with dense, hybrid and hybrid+rerank retrieval, on both question sets.

In [12]:
def hit_hand(item, idx):
    c = CHUNKS[idx]
    return c["topic"] == item["topic"] and any(k in c["text"].lower() for k in item["kw"])

def hit_syn(item, idx):
    return idx == item["chunk_id"]

def eval_retrieval(kb, items, hit_fn, mode, k=3):
    hits, rr = 0, []
    for it in items:
        ranked = [r["idx"] for r in retrieve(it["q"], kb, mode, k_final=k)]
        pos = next((i for i, idx in enumerate(ranked) if hit_fn(it, idx)), None)
        hits += pos is not None
        rr.append(1.0 / (pos + 1) if pos is not None else 0.0)
    return hits / len(items), float(np.mean(rr))

rows = []
for emb_name, kb in KBS.items():
    for mode in ["dense", "hybrid", "hybrid+rerank"]:
        h1, m1 = eval_retrieval(kb, HAND_EVAL, hit_hand, mode)
        h2, m2 = eval_retrieval(kb, SYNTH, hit_syn, mode)
        rows.append({"embedding": emb_name.split("/")[-1], "retriever": mode,
                     "hand Hit@3": round(h1, 3), "hand MRR": round(m1, 3),
                     "synthetic Hit@3": round(h2, 3), "synthetic MRR": round(m2, 3)})

RETRIEVAL_TABLE = pd.DataFrame(rows)
RETRIEVAL_TABLE.to_csv(os.path.join(CONFIG["WORK_DIR"], "results_retrieval.csv"), index=False)
RETRIEVAL_TABLE

,embedding,retriever,hand Hit@3,hand MRR,synthetic Hit@3,synthetic MRR
0,bge-small-en-v1.5,dense,1.0,0.955,1.000,0.928
1,bge-small-en-v1.5,hybrid,1.0,0.939,0.933,0.822
2,bge-small-en-v1.5,hybrid+rerank,1.0,0.985,0.967,0.844
3,bge-base-en-v1.5,dense,1.0,0.970,0.967,0.911
4,bge-base-en-v1.5,hybrid,1.0,0.904,0.933,0.806
5,bge-base-en-v1.5,hybrid+rerank,1.0,0.985,0.967,0.844


### Refusal behaviour (gate only, no LLM): threshold was calibrated on separate questions

In [13]:
rows = []
for emb_name, kb in KBS.items():
    refused = np.mean([best_score(q, kb) < CONFIG["RERANK_THRESHOLD"] for q in OUT_EVAL])
    attempted = np.mean([best_score(x["q"], kb) >= CONFIG["RERANK_THRESHOLD"] for x in HAND_EVAL])
    rows.append({"embedding": emb_name.split("/")[-1],
                 "out-of-scope correctly refused": round(float(refused), 3),
                 "in-scope correctly attempted": round(float(attempted), 3)})
REFUSAL_TABLE = pd.DataFrame(rows)
REFUSAL_TABLE.to_csv(os.path.join(CONFIG["WORK_DIR"], "results_refusal.csv"), index=False)
REFUSAL_TABLE

,embedding,out-of-scope correctly refused,in-scope correctly attempted
0,bge-small-en-v1.5,0.625,1.0
1,bge-base-en-v1.5,0.625,1.0


## Step 13: LLM comparison, phase 1 - generate answers

Every candidate LLM answers the same questions using the same retrieval (the default embedding model). Answers, contexts and timings are saved after each model so that progress survives a crash. Re-running skips models already saved.

In [14]:
EVAL_QS = [x["q"] for x in HAND_EVAL] + [x["q"] for x in SYNTH]
ANS_PATH = os.path.join(CONFIG["WORK_DIR"], "answers.json")
ANSWERS = json.load(open(ANS_PATH, encoding="utf-8")) if os.path.exists(ANS_PATH) else {}
kb_default = KBS[CONFIG["DEFAULT_EMBED"]]

for name in CONFIG["LLM_MODELS"]:
    if name in ANSWERS:
        print("Skipping (already done):", name)
        continue
    print("\n=== Generating with", name)
    llm = LocalLLM(name)
    rows = []
    for q in tqdm(EVAL_QS):
        t0 = time.time()
        res = ask(q, kb_default, llm)
        rows.append({"q": q, "answer": res["answer"], "answered": res["answered"],
                     "context": res["context"], "seconds": round(time.time() - t0, 2)})
    llm.close()
    ANSWERS[name] = rows
    json.dump(ANSWERS, open(ANS_PATH, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    print("Saved answers for", name)


=== Generating with Qwen/Qwen2.5-1.5B-Instruct


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

... (17 outputs left out)

Saved answers for Qwen/Qwen2.5-3B-Instruct

=== Generating with Qwen/Qwen2.5-7B-Instruct


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

  0%|          | 0/63 [00:00<?, ?it/s]

Saved answers for Qwen/Qwen2.5-7B-Instruct


## Step 14: LLM comparison, phase 2 - faithfulness with an LLM judge

The judge sees only the context and the answer and labels it `SUPPORTED`, `PARTIAL` or `UNSUPPORTED`. Score: 1 / 0.5 / 0, averaged over answered questions. The same judge is used for every model.

In [15]:
def parse_label(text):
    t = text.upper()
    for lab in ["UNSUPPORTED", "PARTIAL", "SUPPORTED"]:      # UNSUPPORTED first, it contains "SUPPORTED"
        if lab in t:
            return lab
    return "UNSUPPORTED"

JUDGE_PATH = os.path.join(CONFIG["WORK_DIR"], "judged.json")
JUDGED = json.load(open(JUDGE_PATH, encoding="utf-8")) if os.path.exists(JUDGE_PATH) else {}

if any(m not in JUDGED for m in ANSWERS):
    judge = LocalLLM(CONFIG["JUDGE_MODEL"])
    for name, rows in ANSWERS.items():
        if name in JUDGED:
            continue
        print("Judging", name)
        labelled = []
        for r in tqdm(rows):
            r = dict(r)
            if r["answered"]:
                r["label"] = parse_label(judge.chat([{"role": "user", "content":
                    "You are a strict fact-checker.\n\nContext:\n" + r["context"] +
                    "\n\nAnswer:\n" + r["answer"] +
                    "\n\nIs EVERY factual claim in the Answer supported by the Context? "
                    "Reply with exactly one word: SUPPORTED, PARTIAL, or UNSUPPORTED."}], max_new_tokens=6))
            else:
                r["label"] = None
            labelled.append(r)
        JUDGED[name] = labelled
        json.dump(JUDGED, open(JUDGE_PATH, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    judge.close()

score = {"SUPPORTED": 1.0, "PARTIAL": 0.5, "UNSUPPORTED": 0.0}
rows = []
for name, labelled in JUDGED.items():
    ans = [r for r in labelled if r["answered"]]
    labs = [r["label"] for r in ans]
    rows.append({
        "LLM": name.split("/")[-1],
        "questions": len(labelled),
        "answered %": round(100 * len(ans) / len(labelled), 1),
        "faithfulness": round(float(np.mean([score[l] for l in labs])), 3) if labs else None,
        "SUPPORTED": labs.count("SUPPORTED"), "PARTIAL": labs.count("PARTIAL"), "UNSUPPORTED": labs.count("UNSUPPORTED"),
        "avg sec/answer": round(float(np.mean([r["seconds"] for r in labelled])), 1),
    })
LLM_TABLE = pd.DataFrame(rows)
LLM_TABLE.to_csv(os.path.join(CONFIG["WORK_DIR"], "results_llm.csv"), index=False)
LLM_TABLE

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Judging Qwen/Qwen2.5-1.5B-Instruct


  0%|          | 0/63 [00:00<?, ?it/s]

Judging Qwen/Qwen2.5-3B-Instruct


  0%|          | 0/63 [00:00<?, ?it/s]

Judging Qwen/Qwen2.5-7B-Instruct


  0%|          | 0/63 [00:00<?, ?it/s]

,LLM,questions,answered %,faithfulness,SUPPORTED,PARTIAL,UNSUPPORTED,avg sec/answer
0,Qwen2.5-1.5B-Instruct,63,100.0,0.825,44,16,3,6.3
1,Qwen2.5-3B-Instruct,63,96.8,0.918,52,8,1,7.1
2,Qwen2.5-7B-Instruct,63,100.0,0.937,55,8,0,8.4


### Manual spot-check of the judge (recommended for credibility)

An LLM judge can be wrong. This exports 25 random judged answers to a CSV. Download it from the notebook output, fill the `human_label` column (`SUPPORTED`, `PARTIAL` or `UNSUPPORTED`), upload it as a Kaggle dataset, set `MANUAL_CSV`, and run the second cell to get the judge-vs-human agreement.

In [16]:
flat = [dict(model=m.split("/")[-1], question=r["q"], answer=r["answer"], context=r["context"][:1500],
             judge_label=r["label"], human_label="")
        for m, rs in JUDGED.items() for r in rs if r["answered"]]
random.shuffle(flat)
pd.DataFrame(flat[:25]).to_csv(os.path.join(CONFIG["WORK_DIR"], "manual_check.csv"), index=False)
print("Wrote manual_check.csv with", min(25, len(flat)), "rows (see the notebook Output tab).")

Wrote manual_check.csv with 25 rows (see the notebook Output tab).


In [17]:
MANUAL_CSV = ""   # e.g. "/kaggle/input/manual-check/manual_check.csv" after you labelled it

if MANUAL_CSV and os.path.exists(MANUAL_CSV):
    from sklearn.metrics import cohen_kappa_score
    mc = pd.read_csv(MANUAL_CSV).dropna(subset=["human_label"])
    mc["human_label"] = mc["human_label"].str.strip().str.upper()
    agree = (mc["human_label"] == mc["judge_label"]).mean()
    kappa = cohen_kappa_score(mc["human_label"], mc["judge_label"])
    print(f"Rows: {len(mc)} | exact agreement: {agree:.2f} | Cohen's kappa: {kappa:.2f}")
else:
    print("Set MANUAL_CSV after labelling to compute agreement.")

Set MANUAL_CSV after labelling to compute agreement.


## Step 15: Choose the final configuration and export the Space folder

Set the embedding model from your Step 12 table. For the **Space**, a free CPU Space cannot run a 3B-7B 4-bit model, so the default backend is a **hosted LLM through the Hugging Face Inference API** (`hf_inference`); only the embedder, reranker and FAISS run inside the Space. Availability and limits of hosted models change, so pick a model that is currently served, or set `SPACE_BACKEND = "local"` to run a tiny model on CPU (slow, lower quality).

In [18]:
BEST_EMBED = CONFIG["DEFAULT_EMBED"]          # change after reading the Step 12 table
SPACE_BACKEND = "hf_inference"                # "hf_inference" or "local"
SPACE_LLM = "Qwen/Qwen2.5-7B-Instruct"        # hosted model (backend = hf_inference)
SPACE_LOCAL_LLM = "Qwen/Qwen2.5-0.5B-Instruct"  # used when backend = local (CPU)

SPACE_DIR = os.path.join(CONFIG["WORK_DIR"], "space")
os.makedirs(SPACE_DIR, exist_ok=True)

faiss.write_index(KBS[BEST_EMBED].index, os.path.join(SPACE_DIR, "faiss.index"))
json.dump(CHUNKS, open(os.path.join(SPACE_DIR, "chunks.json"), "w", encoding="utf-8"), ensure_ascii=False)
json.dump({
    "embed_model": BEST_EMBED,
    "rerank_model": CONFIG["RERANK_MODEL"],
    "threshold": CONFIG["RERANK_THRESHOLD"],
    "top_k_candidates": CONFIG["TOP_K_CANDIDATES"],
    "top_k_final": CONFIG["TOP_K_FINAL"],
    "backend": SPACE_BACKEND,
    "llm_model": SPACE_LLM if SPACE_BACKEND == "hf_inference" else SPACE_LOCAL_LLM,
    "disclaimer": DISCLAIMER,
}, open(os.path.join(SPACE_DIR, "config.json"), "w"), indent=1)

print("Exported:", os.listdir(SPACE_DIR))

Exported: ['faiss.index', 'chunks.json', 'config.json']


### `app.py` for the Space (written by the next cell)

In [19]:
%%writefile /kaggle/working/space/app.py
import os, re, json
import numpy as np
import faiss
import gradio as gr
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi

BASE = os.path.dirname(os.path.abspath(__file__))
cfg = json.load(open(os.path.join(BASE, "config.json"), encoding="utf-8"))
CHUNKS = json.load(open(os.path.join(BASE, "chunks.json"), encoding="utf-8"))
index = faiss.read_index(os.path.join(BASE, "faiss.index"))

REFUSAL = "I don't know based on the provided documents."
BGE_QUERY_PREFIX = "Represent this sentence for searching relevant passages: "
DISCLAIMER = cfg["disclaimer"]
THRESHOLD = float(os.environ.get("RERANK_THRESHOLD", cfg["threshold"]))


def tokenize(text):
    return re.findall(r"[a-z0-9]+", text.lower())


bm25 = BM25Okapi([tokenize(c["embed_text"]) for c in CHUNKS])
embedder = SentenceTransformer(cfg["embed_model"], device="cpu")
reranker = CrossEncoder(cfg["rerank_model"], device="cpu")

BACKEND = os.environ.get("LLM_BACKEND", cfg["backend"])
LLM_MODEL = os.environ.get("LLM_MODEL", cfg["llm_model"])

if BACKEND == "hf_inference":
    from huggingface_hub import InferenceClient
    _client = InferenceClient(model=LLM_MODEL, token=os.environ.get("HF_TOKEN"))

    def chat(messages, max_new_tokens=300):
        try:
            out = _client.chat_completion(messages=messages, max_tokens=max_new_tokens, temperature=0.01)
            return out.choices[0].message.content.strip()
        except Exception as e:
            return "LLM_ERROR: " + str(e)[:200]
else:
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    _tok = AutoTokenizer.from_pretrained(LLM_MODEL)
    _model = AutoModelForCausalLM.from_pretrained(LLM_MODEL)
    _model.eval()

    def chat(messages, max_new_tokens=300):
        text = _tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = _tok(text, return_tensors="pt")
        with torch.no_grad():
            out = _model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                                  temperature=None, top_p=None, top_k=None,
                                  pad_token_id=_tok.eos_token_id)
        return _tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()


def retrieve(query, k_final=None, k_cand=None):
    k_final = k_final or cfg["top_k_final"]
    k_cand = k_cand or cfg["top_k_candidates"]
    q = embedder.encode([BGE_QUERY_PREFIX + query], normalize_embeddings=True).astype("float32")
    sims, ids = index.search(q, k_cand)
    dense = [int(i) for i in ids[0] if i >= 0]
    bm_ids = np.argsort(-bm25.get_scores(tokenize(query)))[:k_cand]
    rrf = {}
    for rank, i in enumerate(dense):
        rrf[i] = rrf.get(i, 0.0) + 1.0 / (60 + rank + 1)
    for rank, i in enumerate(bm_ids):
        rrf[int(i)] = rrf.get(int(i), 0.0) + 1.0 / (60 + rank + 1)
    fused = sorted(rrf, key=rrf.get, reverse=True)[:k_cand]
    scores = reranker.predict([(query, CHUNKS[i]["embed_text"]) for i in fused])
    order = np.argsort(-scores)[:k_final]
    return [{"idx": fused[o], "score": float(scores[o])} for o in order]


EMERGENCY_PATTERN = re.compile(
    r"chest pain|can'?t breathe|cannot breathe|difficulty breathing|shortness of breath|"
    r"stroke|face droop|sudden weakness|unconscious|seizure|severe bleeding|suicid|overdose", re.I)
EMERGENCY_NOTE = ("**If this is happening to you or someone else right now, contact your local emergency "
                  "number or go to the nearest emergency department immediately.**")

SYSTEM_PROMPT = (
    "You are a careful medical information assistant.\n"
    "Answer the user's question using ONLY the numbered context passages provided.\n"
    "Rules:\n"
    "1. Do not use outside knowledge and do not add facts that are not in the context.\n"
    "2. Be concise: a short paragraph or a few bullet points.\n"
    "3. Mention passage numbers like [1] or [2] after the facts they support.\n"
    "4. If the context does not contain the answer, reply exactly: " + REFUSAL + "\n"
    "5. Never diagnose the user or prescribe doses; suggest talking to a doctor when relevant."
)


def rewrite_question(question, history):
    if not history:
        return question
    convo = "\n".join("User: " + u + "\nAssistant: " + a[:300] for u, a in history[-3:])
    msgs = [
        {"role": "system", "content": "Rewrite the user's last question as a single standalone question that makes "
                                      "sense without the conversation. Replace pronouns like 'it' or 'its' with the "
                                      "topic. If it is already standalone, return it unchanged. Output only the question."},
        {"role": "user", "content": "Conversation:\n" + convo + "\n\nLast question: " + question + "\n\nStandalone question:"},
    ]
    out = chat(msgs, max_new_tokens=60)
    if out.startswith("LLM_ERROR"):
        return question
    return out.split("\n")[0].strip() or question


def ask(question, history):
    standalone = rewrite_question(question, history)
    results = retrieve(standalone)
    best = results[0]["score"] if results else float("-inf")
    emergency = EMERGENCY_PATTERN.search(question) is not None

    if best < THRESHOLD:
        return REFUSAL + ("\n\n" + EMERGENCY_NOTE if emergency else ""), []

    context = "\n\n".join(
        "[" + str(n) + "] (" + CHUNKS[r["idx"]]["org"] + ": " + CHUNKS[r["idx"]]["title"] + " - "
        + CHUNKS[r["idx"]]["heading"] + ")\n" + CHUNKS[r["idx"]]["text"]
        for n, r in enumerate(results, start=1))
    answer = chat([{"role": "system", "content": SYSTEM_PROMPT},
                   {"role": "user", "content": "Context:\n" + context + "\n\nQuestion: " + standalone}])

    if answer.startswith("LLM_ERROR"):
        return "The language model service is temporarily unavailable. Please try again in a moment.", []
    if REFUSAL.lower() in answer.lower() and len(answer) < len(REFUSAL) + 20:
        return REFUSAL + ("\n\n" + EMERGENCY_NOTE if emergency else ""), []

    sources, seen = [], set()
    for n, r in enumerate(results, start=1):
        c = CHUNKS[r["idx"]]
        key = (c["url"], c["heading"])
        if key not in seen:
            seen.add(key)
            sources.append((n, c))
    if emergency:
        answer += "\n\n" + EMERGENCY_NOTE
    return answer, sources


def to_pairs(history):
    pairs = []
    if history and isinstance(history[0], dict):
        user = None
        for m in history:
            content = m["content"] if isinstance(m["content"], str) else str(m["content"])
            if m["role"] == "user":
                user = content
            elif m["role"] == "assistant" and user is not None:
                pairs.append((user, content))
                user = None
    else:
        pairs = [(u, b) for u, b in (history or []) if u and b]
    return pairs


def respond(message, history):
    answer, sources = ask(message, to_pairs(history))
    text = answer
    if sources:
        lines = []
        for n, c in sources:
            label = "[" + str(n) + "] " + c["org"] + ": " + c["title"] + " - " + c["heading"]
            lines.append("- [" + label + "](" + c["url"] + ")" if c["url"].startswith("http") else "- " + label)
        text += "\n\n**Sources**\n" + "\n".join(lines)
    return text + "\n\n_" + DISCLAIMER + "_"


demo = gr.ChatInterface(
    fn=respond,
    title="Medical RAG Chatbot",
    description="Answers only from the public health pages in its knowledge base (MedlinePlus, WHO, NHS, CDC). " + DISCLAIMER,
    examples=["What are the symptoms of diabetes?", "How is high blood pressure treated?",
              "How is malaria transmitted?", "What are the warning signs of a stroke?"],
)

if __name__ == "__main__":
    demo.launch()

Writing /kaggle/working/space/app.py


In [20]:
SPACE_DIR = "/kaggle/working/space"

with open(os.path.join(SPACE_DIR, "requirements.txt"), "w") as f:
    f.write("\n".join(["gradio", "sentence-transformers", "faiss-cpu", "rank_bm25", "numpy",
                       "huggingface_hub", "transformers", "torch"]) + "\n")

with open(os.path.join(SPACE_DIR, "README.md"), "w") as f:
    f.write("---\n"
            "title: Medical RAG Chatbot\n"
            "emoji: 🩺\n"
            "colorFrom: blue\n"
            "colorTo: green\n"
            "sdk: gradio\n"
            "app_file: app.py\n"
            "pinned: false\n"
            "---\n\n"
            "Retrieval-augmented medical information chatbot (MedlinePlus, WHO, NHS, CDC). "
            "Educational project, not medical advice.\n")

import py_compile
py_compile.compile(os.path.join(SPACE_DIR, "app.py"), doraise=True)
print("app.py compiles OK\n")
for fn in sorted(os.listdir(SPACE_DIR)):
    print(f"{fn:20s} {os.path.getsize(os.path.join(SPACE_DIR, fn)) / 1024:9.1f} KB")

app.py compiles OK

README.md                  0.2 KB
__pycache__                4.0 KB
app.py                     7.6 KB
chunks.json              427.9 KB
config.json                0.4 KB
faiss.index              508.5 KB
requirements.txt           0.1 KB
